In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


In [22]:
!cd /tmp/malaika-repo && git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.08 KiB | 550.00 KiB/s, done.
From https://github.com/Vimalk0703/Google-Gemma4-good-hackathon-VimalXMark
   25f89db..76d7dee  main       -> origin/main
Updating 25f89db..76d7dee
Fast-forward
 malaika/evaluation/golden_scenarios.py | 8 +++++++-
 tests/conftest.py                      | 5 +++--
 2 files changed, 10 insertions(+), 3 deletions(-)


In [3]:
!pip install -q git+https://github.com/huggingface/transformers.git structlog "bitsandbytes>=0.46.1" accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.5 MB/s eta 0:00:00:00:01


In [4]:
!pip install -U bitsandbytes>=0.46.1

In [5]:
from huggingface_hub import login                                                                                                                                  
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))         

In [6]:
!git clone -q https://github.com/Vimalk0703/Google-Gemma4-good-hackathon-VimalXMark.git /tmp/malaika-repo
import sys                                                                                                                                                         
sys.path.insert(0, "/tmp/malaika-repo")                                                                                                                            
                                     

In [7]:
import torch                                                                                                                                                       
print(f"GPU: {torch.cuda.get_device_name(0)}")                                                                                                                     
print(f"Capability: {torch.cuda.get_device_capability(0)}")

GPU: Tesla T4
Capability: (7, 5)


In [8]:
import time, json, re, torch                                                                                                                                       
import numpy as np                                                                                                                                                 
from PIL import Image
                                                                                                                                                                 
print(f"GPU: {torch.cuda.get_device_name(0)}")                                                                                                                   
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")  

GPU: Tesla T4
VRAM: 14.6 GB


In [9]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig                                                                          
                                                                                                                                                                 
MODEL_NAME = "google/gemma-4-E4B-it"                                                                                                                               
print(f"Loading {MODEL_NAME} in 4-bit...")                                                                                                                         
t0 = time.monotonic()                                                                                                                                              
                                                                                                                                                                 
processor = AutoProcessor.from_pretrained(MODEL_NAME)                                                                                                              
model = AutoModelForImageTextToText.from_pretrained(                                                                                                               
  MODEL_NAME, device_map="auto",                                                                                                                                 
  quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),                                                             
)                                                                                                                                                                  
load_time = time.monotonic() - t0
print(f"Loaded in {load_time:.0f}s")    

Loading google/gemma-4-E4B-it in 4-bit...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Loaded in 130s


In [10]:
def ask(messages, max_tokens=200):                                                                                                                                 
  fmt = []                                                                                                                                                     
  for m in messages:                                                                                                                                             
      if isinstance(m["content"], str):
          fmt.append({"role": m["role"], "content": [{"type": "text", "text": m["content"]}]})                                                                   
      else:                                                                                                                                                      
          fmt.append(m)
  inputs = processor.apply_chat_template(                                                                                                                        
      fmt, tokenize=True, return_dict=True,                                                                                                                    
      return_tensors="pt", add_generation_prompt=True,                                                                                                           
  ).to(model.device)                                                                                                                                             
  t = time.monotonic()
  with torch.inference_mode():                                                                                                                                   
      out = model.generate(**inputs, max_new_tokens=max_tokens)                                                                                                
  new = out[0][inputs["input_ids"].shape[1]:]                                                                                                                    
  result = processor.decode(new, skip_special_tokens=True)                                                                                                       
  ms = (time.monotonic() - t) * 1000                      
  print(f"  {ms:.0f}ms | {len(new)} tokens | {len(new)/(ms/1000):.1f} tok/s")                                                                                    
  return result                                                                                                                                                
                                                                                                                                                                 
def get_json(text):                                                                                                                                              
  for pattern in [r'\{[^{}]*\}', r'```(?:json)?\s*\n?(.*?)\n?```']:                                                                                              
      m = re.search(pattern, text, re.DOTALL)                                                                                                                  
      if m:                                                                                                                                                      
          try:
              return json.loads(m.group(0) if '{' in m.group(0) else m.group(1))                                                                                 
          except Exception:                                                                                                                                    
              pass                                                                                                                                               
  return None     

In [11]:
img = Image.fromarray(np.random.randint(100, 200, (320, 240, 3), dtype=np.uint8))
img.save("/tmp/test.jpg")                                                                                                                                          
                                                                                                                                                                 
from malaika.prompts import PromptRegistry
from malaika.prompts import breathing, danger_signs, diarrhea, fever, nutrition, treatment                                                                         
from malaika.imci_protocol import classify_breathing, classify_danger_signs, classify_diarrhea, classify_nutrition, classify_assessment                          
                                                                                                                                                                 
print(f"Prompts registered: {len(PromptRegistry.list_all())}")  

Prompts registered: 14


In [12]:
print("TEST 1: DANGER SIGNS — Alertness")                                                                                                                          
p = PromptRegistry.get("danger.assess_alertness")                                                                                                                
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)                                                                                     
j = get_json(r)                                                                                                                                                  
print(f"  JSON: {j}")                                                                                                                                              

TEST 1: DANGER SIGNS — Alertness
  17446ms | 107 tokens | 6.1 tok/s
  JSON: None


In [13]:
print("TEST 2: BREATHING — Chest Indrawing")                                                                                                                       
p = PromptRegistry.get("breathing.detect_chest_indrawing")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)                                                                                     
j = get_json(r)                                                                                                                                                  
print(f"  JSON: {j}")   

TEST 2: BREATHING — Chest Indrawing
  7501ms | 50 tokens | 6.7 tok/s
  JSON: {'indrawing_detected': False, 'confidence': 0.5, 'location': 'none', 'description': 'No discernible chest wall movement visible in the provided static image.'}


In [14]:
print("TEST 3: PROTOCOL — Breathing Classification")                                                                                                               
for age, rate, exp in [(6,55,"pneumonia"), (6,42,"no_pneumonia"), (24,45,"pneumonia"), (24,35,"no_pneumonia"), (10,50,"pneumonia")]:
  c = classify_breathing(age_months=age, breathing_rate=rate, has_cough=True)                                                                                    
  ok = "PASS" if exp in c.classification.value else "FAIL"                                                                                                       
  print(f"  {ok} | Age {age}mo, Rate {rate} -> {c.classification.value} ({c.severity.value})") 

TEST 3: PROTOCOL — Breathing Classification
  PASS | Age 6mo, Rate 55 -> pneumonia (yellow)
  PASS | Age 6mo, Rate 42 -> no_pneumonia_cough_or_cold (green)
  PASS | Age 24mo, Rate 45 -> pneumonia (yellow)
  PASS | Age 24mo, Rate 35 -> no_pneumonia_cough_or_cold (green)
  PASS | Age 10mo, Rate 50 -> pneumonia (yellow)


In [15]:
print("TEST 4: DIARRHEA — Dehydration Signs")                                                                                                                      
p = PromptRegistry.get("diarrhea.assess_dehydration_signs")                                                                                                        
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)                                                                                     
j = get_json(r)                                                                                                                                                  
print(f"  JSON: {j}")   

TEST 4: DIARRHEA — Dehydration Signs
  12980ms | 89 tokens | 6.9 tok/s
  JSON: {'sunken_eyes': False, 'skin_pinch_result': 'not_visible', 'general_appearance': 'normal', 'tears_visible': False, 'dry_mouth_visible': False, 'confidence': 0.3, 'description': 'Image shows a child being assessed for dehydration.'}


In [16]:
print("TEST 5: NUTRITION — Wasting")                                                                                                                               
p = PromptRegistry.get("nutrition.assess_wasting")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)                                                                                     
j = get_json(r)                                                                                                                                                  
print(f"  JSON: {j}") 

TEST 5: NUTRITION — Wasting
  41127ms | 300 tokens | 7.3 tok/s
  JSON: None


In [17]:
print("TEST 6: TREATMENT — Swahili")                                                                                                                             
p = PromptRegistry.get("treatment.generate_plan")
r = ask(p.render(classifications="Pneumonia", urgency="YELLOW", language="Swahili", child_age_months=6), max_tokens=500)                                           
print(f"\n{r}")        

TEST 6: TREATMENT — Swahili
  67331ms | 500 tokens | 7.4 tok/s

```json
{
  "role": "Malaika",
  "instructions": {
    "title": "Matibabu kwa Ajili ya Pneumonia - Chukua Hatua Sasa",
    "main_focus": "Ni programu ya kutibu gharama ndogo hadi gharama za nyumbani.",
    "medication_guide": {
      "ibuprofen": {
        "swahili": "kwa mfano",
        "dosage_info": "kwa tahadhari"
      },
      "antibiotic": {
        "swahili": "kama inavyotolewa kwenye dawa",
        "dosage_info": "kama inavyotolewa kwenye dawa"
      }
    },
    "steps": [
      {
        "step_number": 1,
        "instruction": "Anda dawa za Pneumonia kwa mujibu vya dawa uliyopewa (kwa mfano, dawa za kuzuia maambukizi kama vile cotrimoxazole au dawa za dawa). Soma maelezo ya dawa kwa kina.",
        "focus": "Dawa za kuanza kabla ya kurejea kwa mfumo wa afya."
      },
      {
        "step_number": 2,
        "instruction": "Kumbuka dose za dawa kwa umri wake (6 miezi). Tumia kijenzi chidi kidogo kutumia dawa k

In [26]:
print("TEST 7: GOLDEN SCENARIOS")
from malaika.evaluation.golden_scenarios import GOLDEN_SCENARIOS                                                                                                   
passed = 0                                                                                                                                                       
for s in GOLDEN_SCENARIOS:                                                                                                                                         
  result = classify_assessment(age_months=s.age_months, **s.findings)
  actual = set(c.value for c in result.all_classification_types)                                                                                                 
  expected = set(c.value for c in s.expected_classifications)                                                                                                  
  if expected.issubset(actual) and result.severity == s.expected_severity:                                                                                       
      passed += 1                                                         
      print(f"  PASS | {s.name}")                                                                                                                                
  else:                                                                                                                                                        
      print(f"  FAIL | {s.name}")                                                                                                                                
print(f"\n  Result: {passed}/{len(GOLDEN_SCENARIOS)}")

TEST 7: GOLDEN SCENARIOS
  PASS | danger_lethargic
  PASS | danger_convulsions
  PASS | danger_vomiting_everything
  PASS | breathing_fast_infant
  PASS | breathing_fast_child
  PASS | breathing_severe_pneumonia
  PASS | breathing_normal_cough
  PASS | diarrhea_severe_dehydration
  PASS | diarrhea_some_dehydration
  PASS | diarrhea_no_dehydration
  PASS | diarrhea_dysentery
  PASS | diarrhea_persistent
  PASS | fever_very_severe
  PASS | fever_malaria
  PASS | fever_no_malaria
  PASS | nutrition_severe_wasting
  PASS | nutrition_edema
  PASS | nutrition_normal
  FAIL | healthy_child
  PASS | combined_pneumonia_dehydration
  PASS | combined_severe_pneumonia_malnutrition

  Result: 20/21


In [25]:
print("TEST 8: JSON RELIABILITY")
prompts_to_test = ["danger.assess_alertness", "breathing.detect_chest_indrawing",                                                                                  
  "diarrhea.assess_dehydration_signs", "nutrition.assess_wasting", "nutrition.detect_edema"]
jp = 0                                                                                                                                                             
for name in prompts_to_test:                                                                                                                                       
  p = PromptRegistry.get(name)
  r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)                                                                                 
  j = get_json(r)                                                                                                                                              
  if j:                                                                                                                                                          
      jp += 1
      print(f"  PASS | {name}")                                                                                                                                  
  else:                                                                                                                                                        
      print(f"  FAIL | {name}: {r[:80]}")
print(f"\n  JSON: {jp}/{len(prompts_to_test)}")

TEST 8: JSON RELIABILITY
  10639ms | 73 tokens | 6.9 tok/s
  PASS | danger.assess_alertness
  8190ms | 55 tokens | 6.7 tok/s
  PASS | breathing.detect_chest_indrawing
  16396ms | 117 tokens | 7.1 tok/s
  PASS | diarrhea.assess_dehydration_signs
  12135ms | 83 tokens | 6.8 tok/s
  PASS | nutrition.assess_wasting
  13095ms | 90 tokens | 6.9 tok/s
  PASS | nutrition.detect_edema

  JSON: 5/5


In [20]:
print("=" * 50)                                                                                                                                                    
print("SUMMARY")                                                                                                                                                 
print("=" * 50) 
print(f"Model:    {MODEL_NAME}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")                                                                                                                
print(f"Load:     {load_time:.0f}s")               
print(f"Golden:   {passed}/{len(GOLDEN_SCENARIOS)}")                                                                                                               
print(f"JSON:     {jp}/{len(prompts_to_test)}")                                                                                                                  
print("=" * 50)                                                                                                                                                    
             

SUMMARY
Model:    google/gemma-4-E4B-it
GPU:      Tesla T4
Load:     130s
Golden:   20/21
JSON:     5/5


In [21]:
from malaika.evaluation.golden_scenarios import GOLDEN_SCENARIOS                                                                                                 
                                                                                                                                                                 
for s in GOLDEN_SCENARIOS:                                                                                                                                       
  if s.name == "healthy_child":                                                                                                                                  
      print(f"Findings: {s.findings}")                                                                                                                           
      result = classify_assessment(age_months=s.age_months, **s.findings)
      print(f"Expected: {[c.value for c in s.expected_classifications]}")                                                                                        
      print(f"Actual:   {[c.value for c in result.all_classification_types]}")                                                                                 
      print(f"Expected severity: {s.expected_severity.value}")                                                                                                   
      print(f"Actual severity:   {result.severity.value}")                                                                                                       
      for c in result.classifications:                                                                                                                           
          print(f"  [{c.severity.value}] {c.classification.value}: {c.reasoning}")  

Findings: {'danger_signs': {'lethargic': False, 'unable_to_drink': False, 'convulsions': False}, 'breathing': {'breathing_rate': 28, 'has_cough': False, 'has_indrawing': False, 'has_stridor': False, 'has_wheeze': False}, 'diarrhea': {'has_diarrhea': False}, 'fever': {'has_fever': False}, 'nutrition': {'visible_wasting': False, 'edema': False, 'muac_mm': 150}}
Expected: ['healthy']
Actual:   ['no_pneumonia_cough_or_cold', 'no_malnutrition']
Expected severity: green
Actual severity:   green
  [green] no_pneumonia_cough_or_cold: No pneumonia: no fast breathing, no indrawing. WHO IMCI p.5.
  [green] no_malnutrition: No malnutrition: no wasting, no edema, MUAC 150mm. WHO IMCI p.14.


In [27]:
import importlib                            
import malaika.evaluation.golden_scenarios as gs
importlib.reload(gs)                    
from malaika.evaluation.golden_scenarios import GOLDEN_SCENARIOS                                                                                                   
                                                                                                                                                                 
passed = 0                                                                                                                                                         
for s in GOLDEN_SCENARIOS:                                                                                                                                         
  result = classify_assessment(age_months=s.age_months, **s.findings)                                                                                            
  actual = set(c.value for c in result.all_classification_types)                                                                                                 
  expected = set(c.value for c in s.expected_classifications)                                                                                                  
  if expected.issubset(actual) and result.severity == s.expected_severity:                                                                                       
      passed += 1                                                                                                                                                
      print(f"  PASS | {s.name}")                                                                                                                                
  else:                                                                                                                                                          
      print(f"  FAIL | {s.name}")                                                                                                                                
print(f"\n  Result: {passed}/{len(GOLDEN_SCENARIOS)}")    

  PASS | danger_lethargic
  PASS | danger_convulsions
  PASS | danger_vomiting_everything
  PASS | breathing_fast_infant
  PASS | breathing_fast_child
  PASS | breathing_severe_pneumonia
  PASS | breathing_normal_cough
  PASS | diarrhea_severe_dehydration
  PASS | diarrhea_some_dehydration
  PASS | diarrhea_no_dehydration
  PASS | diarrhea_dysentery
  PASS | diarrhea_persistent
  PASS | fever_very_severe
  PASS | fever_malaria
  PASS | fever_no_malaria
  PASS | nutrition_severe_wasting
  PASS | nutrition_edema
  PASS | nutrition_normal
  PASS | healthy_child
  PASS | combined_pneumonia_dehydration
  PASS | combined_severe_pneumonia_malnutrition

  Result: 21/21
